In [1]:
%load_ext autoreload

%autoreload 2

In [3]:
import pandas as pd
from datetime import datetime, timedelta
from difflib import SequenceMatcher
import matplotlib.pyplot as plt
import os
from pathlib import Path
import seaborn as sns
import numpy as np
from lifelines.utils import to_long_format
from lifelines.utils import add_covariate_to_timeline
from lifelines import CoxTimeVaryingFitter
from statsmodels.distributions.empirical_distribution import ECDF
from scipy.stats import ks_2samp
from scipy import stats
from matplotlib.ticker import MultipleLocator
import matplotlib.ticker as mtick
from lifelines import KaplanMeierFitter
from matching_case_control import *
import warnings
from preproces_prod3 import *
from scipy.stats import norm
import gspread
from oauth2client.service_account import ServiceAccountCredentials
warnings.filterwarnings("ignore")

path_actual = Path.cwd()
path_data = path_actual.parent / 'Data'
path_nirsecl = path_actual.parent.parent/'Nirse_cl' / 'Data'
df = pd.read_csv(path_data/'VRS2022_2023_ENCR.csv', sep=';').assign(
        fecha_nac = lambda x: pd.to_datetime(x.FechaNacimiento, infer_datetime_format=True, errors='coerce'),
        year_nac = lambda x: x.fecha_nac.dt.year,
        month_nac = lambda x: x.fecha_nac.dt.month,)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)

In [4]:
df.head(5)

,RUT,Nº de referencia,Fecha de solicitud,MES Gatilla Solicitud,AÑO Ingreso solicitud,Enfermedad,ESTADO (2),FechaNacimiento,Sexo,previsión,Nombre - Tipo,Nacionalidad,Fecha de confirmación -descarte,Mes Confirmación o descarte,Año Confirmación o descarte,Fecha Cierre de caso,Causal Cierre,Fecha entrega tratamiento,Fecha recepción establecimiento,Causa Excepción,Tipo Beneficiario,RUT_Titular,fecha_nac,year_nac,month_nac
0,c1540362acd37ed6a3020f77243fc42a00f29849a6a4fda2ee78fd4a77e34f38,211006-000083,06-10-2021,Oct2021,2021,Profilaxis VRS,Beneficiarios Inactivos,29-07-2021,Masculino,Fonasa,Tramo: A,CHILE,07-10-2021,Oct2021,2021,22-03-2023,Cumplimiento del Año Cronológico,20-05-2022,08-07-2022,NaN,Carga,cea9bec61cf25108e1726dd712006bfde517f331005f8d99a865caf4fb2d7d9f,2021-07-29,2021,7
1,18f75a1c310856133f81d2ed67611824464454e9d4fafff0e17b3c5c1d5d6949,211027-000397,27-10-2021,Oct2021,2021,Profilaxis VRS,Beneficiarios Inactivos,25-08-2021,Masculino,Isapre,CONSALUD S.A.,CHILE,03-11-2021,Nov2021,2021,22-03-2023,Cumplimiento del Año Cronológico,02-05-2022,21-04-2022,NaN,Carga,e1a81b55d05d88d3f44abe5549dbc750c51f2c7046fb70d47de0545491c74250,2021-08-25,2021,8
2,fd5f656455cfe50b34895d0dea2f866b037b4e6c8b9177e7fcf8579f68f5b979,211023-000025,23-10-2021,Oct2021,2021,Profilaxis VRS,Beneficiarios Inactivos,26-07-2021,Masculino,Isapre,CONSALUD S.A.,CHILE,26-10-2021,Oct2021,2021,22-03-2023,Cumplimiento del Año Cronológico,06-06-2022,21-04-2022,NaN,Carga,67af920ae171726e17788d0c3a1237649b8dead46feac7a6b222a0a8ae1619f1,2021-07-26,2021,7
3,1c82386d356e71b2fbd3fe89bbc0e1124896a81c62a02dd094f39224cbd2460f,211001-000035,01-10-2021,Oct2021,2021,Profilaxis VRS,Beneficiarios Inactivos,16-08-2021,Femenino,Fonasa,Tramo: B,CHILE,03-10-2021,Oct2021,2021,22-03-2023,Cumplimiento del Año Cronológico,04-05-2022,07-06-2022,NaN,Carga,43315f0ff92f2d3f4fccbb530fa071edcc383d27efbeb01099fbe59534c85d95,2021-08-16,2021,8
4,9d56453615f4aef4ecda136e6c3ccb0cb8c9cf090e9f453e86ccf0fdafd2e4a9,211014-000026,14-10-2021,Oct2021,2021,Profilaxis VRS,Beneficiarios Inactivos,20-08-2021,Masculino,Fonasa,Tramo: A,CHILE,14-10-2021,Oct2021,2021,22-03-2023,Cumplimiento del Año Cronológico,02-05-2022,21-04-2022,NaN,Carga,b35a85169f283e8f12bd5025250da3634b54bfa9cf5d8798a311b4c396e70f13,2021-08-20,2021,8


In [5]:
df.columns

Index(['RUT', 'Nº de referencia', 'Fecha de solicitud',
       'MES Gatilla Solicitud', 'AÑO Ingreso solicitud', 'Enfermedad',
       'ESTADO (2)', 'FechaNacimiento', 'Sexo', 'previsión', 'Nombre - Tipo',
       'Nacionalidad', 'Fecha de confirmación -descarte',
       'Mes Confirmación o descarte', 'Año Confirmación o descarte',
       'Fecha Cierre de caso ', 'Causal Cierre', 'Fecha entrega tratamiento',
       'Fecha recepción establecimiento', 'Causa Excepción',
       'Tipo Beneficiario', 'RUT_Titular', 'fecha_nac', 'year_nac',
       'month_nac'],
      dtype='object')

In [6]:
columns= ['RUT', 'Fecha de solicitud','MES Gatilla Solicitud', 'AÑO Ingreso solicitud',
       'ESTADO (2)', 'FechaNacimiento', 'Sexo', 'previsión', 'Nombre - Tipo',
       'Nacionalidad', 'Fecha de confirmación -descarte',
       'Mes Confirmación o descarte', 'Año Confirmación o descarte',
       'Fecha Cierre de caso ', 'Causal Cierre', 'Fecha entrega tratamiento',
       'Fecha recepción establecimiento', 'Causa Excepción',
       'Tipo Beneficiario', 'RUT_Titular', 'fecha_nac', 'year_nac',
       'month_nac']

df = (df[columns]
      .assign(fecha_treat = lambda x: pd.to_datetime(x['Fecha entrega tratamiento'], infer_datetime_format=True, errors='coerce'),
              year_treat = lambda x: x.fecha_treat.dt.year,
              month_treat = lambda x: x.fecha_treat.dt.month,
              fecha_treat_2 = lambda x: pd.to_datetime(x['Fecha recepción establecimiento'], infer_datetime_format=True, errors='coerce'),
              year_treat_2 = lambda x: x.fecha_treat_2.dt.year,
              month_treat_2 = lambda x: x.fecha_treat_2.dt.month,
              year_treat_any = lambda x: np.where(x.year_treat.isna(), x.year_treat_2, x.year_treat),)
)

In [7]:
df.head(5)

,RUT,Fecha de solicitud,MES Gatilla Solicitud,AÑO Ingreso solicitud,ESTADO (2),FechaNacimiento,Sexo,previsión,Nombre - Tipo,Nacionalidad,Fecha de confirmación -descarte,Mes Confirmación o descarte,Año Confirmación o descarte,Fecha Cierre de caso,Causal Cierre,Fecha entrega tratamiento,Fecha recepción establecimiento,Causa Excepción,Tipo Beneficiario,RUT_Titular,fecha_nac,year_nac,month_nac,fecha_treat,year_treat,month_treat,fecha_treat_2,year_treat_2,month_treat_2,year_treat_any
0,c1540362acd37ed6a3020f77243fc42a00f29849a6a4fda2ee78fd4a77e34f38,06-10-2021,Oct2021,2021,Beneficiarios Inactivos,29-07-2021,Masculino,Fonasa,Tramo: A,CHILE,07-10-2021,Oct2021,2021,22-03-2023,Cumplimiento del Año Cronológico,20-05-2022,08-07-2022,NaN,Carga,cea9bec61cf25108e1726dd712006bfde517f331005f8d99a865caf4fb2d7d9f,2021-07-29,2021,7,2022-05-20,2022.0,5.0,2022-08-07,2022.0,8.0,2022.0
1,18f75a1c310856133f81d2ed67611824464454e9d4fafff0e17b3c5c1d5d6949,27-10-2021,Oct2021,2021,Beneficiarios Inactivos,25-08-2021,Masculino,Isapre,CONSALUD S.A.,CHILE,03-11-2021,Nov2021,2021,22-03-2023,Cumplimiento del Año Cronológico,02-05-2022,21-04-2022,NaN,Carga,e1a81b55d05d88d3f44abe5549dbc750c51f2c7046fb70d47de0545491c74250,2021-08-25,2021,8,2022-05-02,2022.0,5.0,NaT,NaN,NaN,2022.0
2,fd5f656455cfe50b34895d0dea2f866b037b4e6c8b9177e7fcf8579f68f5b979,23-10-2021,Oct2021,2021,Beneficiarios Inactivos,26-07-2021,Masculino,Isapre,CONSALUD S.A.,CHILE,26-10-2021,Oct2021,2021,22-03-2023,Cumplimiento del Año Cronológico,06-06-2022,21-04-2022,NaN,Carga,67af920ae171726e17788d0c3a1237649b8dead46feac7a6b222a0a8ae1619f1,2021-07-26,2021,7,2022-06-06,2022.0,6.0,NaT,NaN,NaN,2022.0
3,1c82386d356e71b2fbd3fe89bbc0e1124896a81c62a02dd094f39224cbd2460f,01-10-2021,Oct2021,2021,Beneficiarios Inactivos,16-08-2021,Femenino,Fonasa,Tramo: B,CHILE,03-10-2021,Oct2021,2021,22-03-2023,Cumplimiento del Año Cronológico,04-05-2022,07-06-2022,NaN,Carga,43315f0ff92f2d3f4fccbb530fa071edcc383d27efbeb01099fbe59534c85d95,2021-08-16,2021,8,2022-05-04,2022.0,5.0,2022-07-06,2022.0,7.0,2022.0
4,9d56453615f4aef4ecda136e6c3ccb0cb8c9cf090e9f453e86ccf0fdafd2e4a9,14-10-2021,Oct2021,2021,Beneficiarios Inactivos,20-08-2021,Masculino,Fonasa,Tramo: A,CHILE,14-10-2021,Oct2021,2021,22-03-2023,Cumplimiento del Año Cronológico,02-05-2022,21-04-2022,NaN,Carga,b35a85169f283e8f12bd5025250da3634b54bfa9cf5d8798a311b4c396e70f13,2021-08-20,2021,8,2022-05-02,2022.0,5.0,NaT,NaN,NaN,2022.0


In [8]:
na_counts = df.isna().mean().reset_index()
na_counts.columns = ['Columna', 'prop_Na']
na_counts['prop_Na'] = (round(na_counts['prop_Na'],2) * 100)
na_counts.query('prop_Na > 0')#.sort_values('prop_Na', ascending=False)

,Columna,prop_Na
13,Fecha Cierre de caso,38.0
14,Causal Cierre,38.0
17,Causa Excepción,95.0
23,fecha_treat,34.0
24,year_treat,34.0
25,month_treat,34.0
26,fecha_treat_2,49.0
27,year_treat_2,49.0
28,month_treat_2,49.0
29,year_treat_any,20.0


In [9]:
df.year_treat.value_counts()

year_treat
2022.0    1838
2023.0    1207
Name: count, dtype: int64

In [10]:
df.year_treat_2.value_counts()

year_treat_2
2023.0    1215
2022.0    1141
Name: count, dtype: int64

In [22]:
df_treat = pd.DataFrame(df.year_treat_any.value_counts()).reset_index()
df_treat.columns = ['year','tratados']
df_treat

,year,tratados
0,2022.0,1889
1,2023.0,1787


In [19]:
df_nacimientos = pd.DataFrame({
    'year': [2018, 2019, 2020, 2021, 2022, 2023, 2024],
    'nacimientos': [1811, 3049, 2924, 2839, 3012, 3049, 2792]
})
df_nacimientos_2223 = df_nacimientos.query('year==2022 | year==2023').reset_index(drop=True)
df_nacimientos_2223

,year,nacimientos
0,2022,3012
1,2023,3049


In [27]:
df_merged = pd.merge(df_treat,df_nacimientos_2223, how='left', on='year')
df_prop = (
    df_merged
    .assign(prop = lambda x: round(x.tratados / x.nacimientos, 3))
)
df_prop

,year,tratados,nacimientos,prop
0,2022.0,1889,3012,0.627
1,2023.0,1787,3049,0.586


In [23]:
df.year_nac.value_counts()

year_nac
2022    2503
2023    1643
2021     442
2020       1
1989       1
Name: count, dtype: int64